## 분석 질문
고객별 평균 주문 금액은 얼마인가
## 분석 범위
완료된 주문만
## 필요한 DataFrame
orders , order_items
## 필요한 컬럼
orders: order_id, customer_id, order_status
order_items: order_id, quantity, unit_price
## 각 데이터에서 한 행의 의미
orders 한 행 = 주문 1건
order_items 한 행 = 그 주문에 담긴 상품 1종류
## 필터 조건
완료된 주문만 거름
## 파생 컬럼
order_items에 금액 = quantity * unit_price
## 병합 키와 관계 수
1:N
## 그룹바이 기준
groupby("customer_id")
## 집계 지표
먼저 주문 하나하나의 총액을 구함
그 다음 고객별로, 그 주문 총액들의 평균을 구함
## 검증 방법
직접 그 사람 주문 총액들을 나열해보고 평균을 계산 → 코드 결과랑 일치하는지 대조
## 결과 한 행의 의미
고객 한 명 + 그 고객의 완료 주문 평균 금액

In [3]:
from course_utils import get_project_root, get_data_dir
import pandas as pd

orders = pd.read_csv(get_data_dir()/"raw"/"orders.csv")

print(orders["order_status"].value_counts())

order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64


In [5]:
order_items = pd.read_csv(get_data_dir()/"raw"/"order_items.csv")
print(order_items.columns.tolist())
print(order_items.head(3))

['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']
   order_item_id  order_id  product_id  quantity  unit_price
0              1         1         100         3      102000
1              2         1          87         5       25000
2              3         1           7         3      142000


In [8]:
order_items["item_total"] = order_items["quantity"] * order_items["unit_price"]

In [9]:
print(order_items.head(3))

   order_item_id  order_id  product_id  quantity  unit_price  item_total
0              1         1         100         3      102000      306000
1              2         1          87         5       25000      125000
2              3         1           7         3      142000      426000


In [10]:
completed_orders = orders[orders["order_status"] == "completed"]

In [12]:
order_totals = order_items.groupby("order_id")["item_total"].sum().reset_index()

In [13]:
print(order_totals.head(3))

   order_id  item_total
0         1     1436000
1         2      756000
2         3     1435000


In [14]:
merged = completed_orders.merge(order_totals, on="order_id", how="left")
print(merged.head(3))

   order_id  customer_id  order_date payment_method order_status  item_total
0         1          123  2026-05-07           card    completed     1436000
1         6           87  2026-03-21      naver_pay    completed     1309000
2         9          145  2026-01-20           card    completed      236000


In [15]:
print(len(completed_orders), len(merged))

184 184


In [16]:
customer_avg = merged.groupby("customer_id")["item_total"].mean().reset_index()
customer_avg = customer_avg.rename(columns={"item_total": "avg_order_amount"})
print(customer_avg.head(10))

   customer_id  avg_order_amount
0            3         1589000.0
1            4          603000.0
2            5         1002000.0
3            6          674500.0
4            7         1173000.0
5            8          766000.0
6            9          566000.0
7           12          396000.0
8           13         1549000.0
9           14          905000.0


판매량과 매출 순위 비교

In [17]:
product_stats = order_items.groupby("product_id").agg(
    quantity_sold=("quantity", "sum"),
    total_sales=("item_total", "sum"),
    order_count=("order_id", "nunique")
).reset_index()

print(product_stats.head(10))

   product_id  quantity_sold  total_sales  order_count
0           1             13      2080000            4
1           2             18       612000            9
2           3             18      2736000            6
3           4             17      1190000            7
4           5             17      3162000            5
5           6             23       115000            8
6           7             16      2272000            6
7           8              4       756000            3
8           9             25      4825000            7
9          10             30      1650000            9


잘못된 병합 만들고 진단하기


In [23]:
products = pd.read_csv(get_data_dir()/"raw"/"products.csv")
print(products.columns.tolist())
print(products.head(3))

['product_id', 'product_name', 'category', 'price']
   product_id product_name category   price
0           1  전자기기 상품 001     전자기기  160000
1           2    도서 상품 002       도서   34000
2           3  전자기기 상품 003     전자기기  152000


In [24]:
products_bad = pd.concat(

    [products, products.head(1)],

    ignore_index=True,

)

In [20]:
order_items_work = order_items.copy()

In [25]:
order_items_work.merge(
    products_bad,
    on="product_id",
    how="left",
    validate="many_to_one",
)


MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
  product_id
          1 ...

In [ ]:
고객 개인정보 최소화

In [26]:
customers = pd.read_csv(get_data_dir()/"raw"/"customers.csv")
print(customers.columns.tolist())
print(customers.head(3))

['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
   customer_id name gender  age city signup_date
0            1  김수민      F   19   광주  2024-06-19
1            2  김정호      F   32   대구  2025-11-02
2            3  이경수      F   61   성남  2024-06-12


In [27]:
import pandas as pd
from course_utils import get_data_dir

orders = pd.read_csv(get_data_dir()/"raw"/"orders.csv")
order_items = pd.read_csv(get_data_dir()/"raw"/"order_items.csv")
customers = pd.read_csv(get_data_dir()/"raw"/"customers.csv")


In [28]:
order_items["item_total"] = order_items["quantity"] * order_items["unit_price"]

In [29]:
items_with_customer = order_items.merge(
    orders[["order_id", "customer_id"]],
    on="order_id",
    how="left"
)

In [30]:
customer_sales = items_with_customer.groupby("customer_id").agg(
    total_sales=("item_total", "sum"),
    order_count=("order_id", "nunique"),
    quantity_sold=("quantity", "sum")
).reset_index()

In [31]:
customers_minimal = customers[["customer_id", "gender", "age", "city"]]

In [32]:
result = customers_minimal.merge(customer_sales, on="customer_id", how="left")

print(result.head(10))
print(len(result))

   customer_id gender  age city  total_sales  order_count  quantity_sold
0            1      F   19   광주          NaN          NaN            NaN
1            2      F   32   대구          NaN          NaN            NaN
2            3      F   61   성남    4668000.0          3.0           35.0
3            4      F   55   울산     603000.0          1.0            6.0
4            5      F   19   부산    2004000.0          2.0           13.0
5            6      F   32   성남    3039000.0          3.0           29.0
6            7      F   53   인천    1173000.0          1.0            9.0
7            8      M   32   수원    2626000.0          3.0           29.0
8            9      M   69   서울    1132000.0          2.0           10.0
9           10      F   62   울산          NaN          NaN            NaN
150


In [33]:
print(order_items["item_total"].sum())
print(result["total_sales"].sum())

255610000
255610000.0
